# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 4, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [4]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type


## 4. Schema evolution (opt-in)

In [5]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [6]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## 5b. Bằng chứng: nội dung `_delta_log/` (thêm bởi học viên)

In [7]:
import json
from pathlib import Path

log_dir = Path(table_path) / "_delta_log"
print("Files in _delta_log/:")
for f in sorted(log_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} B)")

for name in ["00000000000000000000.json", "00000000000000000001.json"]:
    print(f"\n===== {name} =====")
    for line in (log_dir / name).read_text(encoding="utf-8").splitlines():
        action = json.loads(line)
        key = next(iter(action))
        body = action[key]
        if key == "metaData":
            body = {**body, "schemaString": json.loads(body["schemaString"])}
        if key == "add":
            body = {k: body[k] for k in ("path", "size", "dataChange", "stats") if k in body}
        print(f"[{key}]")
        print(json.dumps(body, indent=2, ensure_ascii=False)[:1500])

print("\nSchema after evolution:")
print(DeltaTable(table_path).schema())

Files in _delta_log/:
  00000000000000000000.json  (1409 B)
  00000000000000000001.json  (1466 B)

===== 00000000000000000000.json =====
[commitInfo]
{
  "timestamp": 1791129070260,
  "operation": "WRITE",
  "operationParameters": {
    "mode": "Overwrite"
  },
  "engineInfo": "delta-rs:py-1.6.6",
  "clientVersion": "delta-rs.py-1.6.6",
  "operationMetrics": {
    "num_added_files": 1,
    "num_removed_files": 0,
    "num_partitions": 0,
    "num_added_rows": 3,
    "execution_time_ms": 4,
    "num_retries": 0
  }
}
[protocol]
{
  "minReaderVersion": 1,
  "minWriterVersion": 2
}
[metaData]
{
  "id": "d4b92e3a-8fa2-4abd-8361-c17e1250f885",
  "name": null,
  "description": null,
  "format": {
    "provider": "parquet",
    "options": {}
  },
  "schemaString": {
    "type": "struct",
    "fields": [
      {
        "name": "id",
        "type": "long",
        "nullable": true,
        "metadata": {}
      },
      {
        "name": "name",
        "type": "string",
        "nullable": tr

## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The final schema-enforcement flag is hardcoded; inspect the actual error
from the bad-write cell rather than treating that PASS line as proof.

In [8]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": True,   # placeholder; inspect the bad-write output
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits
  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.


## 📝 Giải thích kết quả (học viên)
**Số đo trên máy tôi:** `_delta_log/` có 2 commit JSON (`…000.json` 1409 B, `…001.json` 1466 B). Ghi `age='thirty'` bị chặn với lỗi thật
`Cast error: Cannot cast string 'thirty' to value of Int64 type`. Sau `schema_mode="merge"` schema có thêm `tier: string`, DuckDB trả 2 nhóm: `premium` (1) và `NULL` (3).

**Transaction log chứng minh gì về một lần ghi?** Mỗi commit là một file JSON gồm các action: `commitInfo` (ai/khi nào/operation, metrics như `num_added_rows`),
`protocol`, `metaData` (schema hiện hành) và `add` (đường dẫn file Parquet + size + stats min/max/nullCount). Bảng ở version N chính là tập hợp các `add` chưa bị `remove`
tính tới commit N. Vì file JSON được tạo nguyên tử (put-if-absent), một lần ghi hoặc hiện diện đầy đủ, hoặc không tồn tại → đó là nền của ACID.
Commit 1 chứa `metaData` mới có thêm field `tier` → evolution cũng được ghi lại như một transaction.

**Enforcement vs evolution:** enforcement là mặc định — writer so schema dữ liệu mới với `metaData` trong log và từ chối nếu không cast được (lần ghi sai không tạo commit nào, bảng vẫn có 2 file JSON).
Evolution là *thay đổi* schema đó, nên phải opt-in (`schema_mode="merge"`): nếu tự động chấp nhận mọi cột/kiểu lạ, một upstream lỗi (đổi tên cột, gửi string thay int)
sẽ âm thầm làm bẩn bảng và làm hỏng mọi consumer phía sau. Các dòng cũ đọc `tier = NULL` mà không cần rewrite file.

*Lưu ý:* cờ `schema enforcement blocked bad write` ở cell cuối là hardcoded `True`; bằng chứng thật là output lỗi của cell ghi sai kiểu ở trên.
